# P3 — Experimentación con MLflow

**RA/CE**: RA4c (modelos de automatización), RA4d (evaluar conveniencia)
**Fase**: F3 — Experimentación y Registro
**Teoría asociada**: `01-teoria/04_experimentacion_mlflow.md`

---

## Objetivos de Aprendizaje

1. Registrar experimentos completos con MLflow Tracking (parámetros, métricas, artefactos)
2. Comparar runs para seleccionar el mejor modelo basándose en métricas objetivas
3. Vincular cada run con la versión de datos (hash DVC) para trazabilidad
4. Utilizar el Model Registry para versionar y promover modelos a producción
5. Evaluar la conveniencia de cada modelo (RA4d) según múltiples criterios

## Prerrequisitos

- [ ] MLflow básico: `log_param`, `log_metric`, `log_artifact` (visto en UD5/UD6)
- [ ] Finalizada la práctica P2 (tienes los datos procesados y los splits)
- [ ] Datos de entrenamiento en `02-ejemplos/pipeline_completo/data/splits/`
- [ ] Scikit-learn y XGBoost instalados

## Contexto

En P2 construiste un pipeline de datos que produjo features listos para entrenar.
Ahora vas a entrenar **3 modelos distintos** para clasificar incidencias técnicas,
registrar cada experimento con MLflow, comparar resultados y seleccionar el mejor.

---
## Parte 1: Verifica tu entorno

In [ ]:
import sys, pickle, json, pathlib, subprocess

HAS_DEPS = True
missing = []

try:
    import mlflow
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    print('✅ Core dependencies OK')
except ImportError as e:
    HAS_DEPS = False
    mod = str(e).split("'")[1]
    missing.append(mod)
    print(f'❌ Missing: {mod}')

try:
    import xgboost as xgb
    print('✅ XGBoost OK')
except ImportError:
    print('⚠️ XGBoost not installed. Install: pip install xgboost')

if not HAS_DEPS:
    print(f'\nInstall: pip install {" ".join(missing)}')
else:
    print('\n✅ Environment ready')

---
## Parte 2: Cargar datos del pipeline

Cargamos los splits generados en P2. Si no existen, los generamos desde cero.

In [ ]:
# Cargar splits del pipeline
BASE = pathlib.Path("..") / "02-ejemplos" / "pipeline_completo"
BASE = BASE.resolve()
SPLITS_DIR = BASE / "data" / "splits"

if SPLITS_DIR.exists() and any(SPLITS_DIR.iterdir()):
    with open(SPLITS_DIR / "X_train.pkl", "rb") as f:
        X_train = pickle.load(f)
    with open(SPLITS_DIR / "X_test.pkl", "rb") as f:
        X_test = pickle.load(f)
    with open(SPLITS_DIR / "y_train.pkl", "rb") as f:
        y_train = pickle.load(f)
    with open(SPLITS_DIR / "y_test.pkl", "rb") as f:
        y_test = pickle.load(f)
    print(f'✅ Loaded pipeline splits: {X_train.shape[0]} train, {X_test.shape[0]} test')
else:
    print('⚠️ Splits not found. Generating sample data for demo...')
    # Generate sample data for standalone execution
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.model_selection import train_test_split
    
    descriptions = [
        'El servidor no responde desde las 14:00',
        'Error de autenticacion en el login',
        'La base de datos esta muy lenta en consultas',
        'Intento de acceso no autorizado detectado',
        'El certificado SSL ha expirado',
        'El balanceador no distribuye el trafico',
        'La aplicacion se congela al cargar datos',
        'Error 500 al procesar el formulario',
        'El disco duro esta al 95%% de capacidad',
        'La memoria RAM se satura con carga normal',
    ] * 20
    categories = (
        ['infraestructura'] * 40 +
        ['seguridad'] * 40 +
        ['rendimiento'] * 40 +
        ['software'] * 40 +
        ['hardware'] * 40
    )
    
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    X = vectorizer.fit_transform(descriptions)
    y = pd.Series(categories)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f'✅ Sample data created: {X_train.shape[0]} train, {X_test.shape[0]} test')

---
## Parte 3: Configurar MLflow

Configuramos MLflow para que guarde los experimentos localmente.

In [ ]:
# Configurar MLflow
MLFLOW_DIR = BASE / "mlruns"
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")

# Crear o reutilizar el experimento
EXPERIMENT_NAME = "ticket_classifier"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print(f'✅ Experiment "{EXPERIMENT_NAME}" ready')
print(f'   Tracking URI: {mlflow.get_tracking_uri()}')
print(f'   Experiment ID: {experiment.experiment_id}')

---
## Parte 4: Entrenar y registrar modelos

Vamos a entrenar 3 modelos distintos y registrar cada uno como una run de MLflow.

In [ ]:
# Obtener version de datos (si DVC instalado)
data_version = "unknown"
try:
    result = subprocess.run(
        ["dvc", "hash", str(BASE / "data" / "raw" / "tickets.csv")],
        capture_output=True, text=True, timeout=5
    )
    if result.returncode == 0:
        data_version = result.stdout.strip().split()[0]
except Exception:
    pass

print(f'📊 Data version: {data_version}')

# Función helper para evaluar
def evaluate_model(model, X_test, y_test):
    """Evaluate model and return metrics dict."""
    y_pred = model.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred, average='weighted'),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
    }

### 4.1 Logistic Regression

In [ ]:
with mlflow.start_run(run_name="logistic_regression"):
    # Parámetros
    params = {
        "model_type": "LogisticRegression",
        "C": 1.0,
        "max_iter": 1000,
        "solver": "liblinear",
        "data_version": data_version,
    }
    mlflow.log_params(params)
    
    # Entrenar
    model = LogisticRegression(C=1.0, max_iter=1000, solver='liblinear')
    model.fit(X_train, y_train)
    
    # Evaluar
    metrics = evaluate_model(model, X_test, y_test)
    mlflow.log_metrics(metrics)
    
    # Guardar modelo
    mlflow.sklearn.log_model(model, "model")
    
    print(f'✅ Logistic Regression: accuracy={metrics["accuracy"]:.3f}, '
          f'f1={metrics["f1_score"]:.3f}')

### 4.2 Random Forest

In [ ]:
with mlflow.start_run(run_name="random_forest"):
    # Parámetros
    params = {
        "model_type": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 5,
        "data_version": data_version,
    }
    mlflow.log_params(params)
    
    # Entrenar
    model = RandomForestClassifier(
        n_estimators=100, max_depth=10,
        min_samples_split=5, random_state=42
    )
    model.fit(X_train, y_train)
    
    # Evaluar
    metrics = evaluate_model(model, X_test, y_test)
    mlflow.log_metrics(metrics)
    
    # Guardar modelo
    mlflow.sklearn.log_model(model, "model")
    
    print(f'✅ Random Forest: accuracy={metrics["accuracy"]:.3f}, '
          f'f1={metrics["f1_score"]:.3f}')

### 4.3 XGBoost

In [ ]:
try:
    import xgboost as xgb
    
    with mlflow.start_run(run_name="xgboost"):
        # Parámetros
        params = {
            "model_type": "XGBClassifier",
            "n_estimators": 100,
            "learning_rate": 0.1,
            "max_depth": 6,
            "subsample": 0.8,
            "data_version": data_version,
        }
        mlflow.log_params(params)
        
        # Entrenar
        model = xgb.XGBClassifier(
            n_estimators=100, learning_rate=0.1,
            max_depth=6, subsample=0.8, random_state=42
        )
        model.fit(X_train, y_train)
        
        # Evaluar
        metrics = evaluate_model(model, X_test, y_test)
        mlflow.log_metrics(metrics)
        
        # Guardar modelo
        mlflow.sklearn.log_model(model, "model")
        
        print(f'✅ XGBoost: accuracy={metrics["accuracy"]:.3f}, '
              f'f1={metrics["f1_score"]:.3f}')
except ImportError:
    print('⚠️ XGBoost no disponible, saltando este modelo...')

---
## Parte 5: Comparación de resultados (RA4d)

Recuperamos todas las runs del experimento y las comparamos.

In [ ]:
# Recuperar todas las runs del experimento
client = mlflow.tracking.MlflowClient()
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_score DESC"]
)

print(f"📊 Comparación de modelos - Experimento: {EXPERIMENT_NAME}\n")
print(f"{'Run':<25} {'Accuracy':<10} {'F1':<10} {'Precision':<10} {'Recall':<10} {'Modelo':<15}")
print("-" * 80)

for run in runs:
    name = run.data.tags.get('mlflow.runName', 'unknown')
    acc = run.data.metrics.get('accuracy', 0)
    f1 = run.data.metrics.get('f1_score', 0)
    prec = run.data.metrics.get('precision', 0)
    rec = run.data.metrics.get('recall', 0)
    model_type = run.data.params.get('model_type', 'unknown')
    print(f"{name:<25} {acc:<10.3f} {f1:<10.3f} {prec:<10.3f} {rec:<10.3f} {model_type:<15}")

### Selección del mejor modelo

Basándote en la tabla anterior:

1. ¿Qué modelo tiene mejor F1-score? ¿Y mejor accuracy?
2. ¿Son consistentes ambas métricas o hay diferencias?
3. Además de accuracy/F1, ¿qué otros criterios considerarías para elegir
   el modelo a producción?

**Conexión RA4d**: Evaluar la conveniencia de cada modelo implica considerar:
- Rendimiento (métricas)
- Tiempo de entrenamiento
- Tamaño del modelo (latencia en inferencia)
- Interpretabilidad (¿puedes explicar por qué predijo X?)
- Mantenibilidad (¿es fácil de actualizar?)

**Tu análisis**:
```
Modelo seleccionado: __________________________________
Criterios usados: ______________________________________
Justificación: ________________________________________
```

---
## Parte 6: Model Registry

Registramos el mejor modelo en el Model Registry y lo promovemos a Staging.

In [ ]:
# Obtener la mejor run (ya ordenada por F1 descendente)
best_run = runs[0] if runs else None

if best_run:
    best_run_id = best_run.info.run_id
    best_name = best_run.data.tags.get('mlflow.runName', 'unknown')
    best_f1 = best_run.data.metrics.get('f1_score', 0)
    print(f'🏆 Mejor modelo: {best_name} (F1={best_f1:.3f})')
    print(f'   Run ID: {best_run_id}')
    
    # Registrar en Model Registry
    model_uri = f"runs:/{best_run_id}/model"
    model_name = "TicketClassifier"
    
    try:
        registered = mlflow.register_model(model_uri, model_name)
        print(f'✅ Modelo registrado: {model_name} (v{registered.version})')
        
        # Promover a Staging
        client.transition_model_version_stage(
            name=model_name,
            version=registered.version,
            stage="Staging"
        )
        print(f'✅ Modelo promovido a Staging')
        print(f'\n📌 Para servir el modelo:')
        print(f'   mlflow models serve -m "models:/{model_name}/Staging" -p 5000')
    except Exception as e:
        print(f'❌ Error registrando modelo: {e}')
        print('   (El Model Registry requiere servidor MLflow, no solo tracking local)')
else:
    print('❌ No hay runs para registrar')

---
## Parte 7: Visualización (MLflow UI)

Para ver los experimentos en la interfaz gráfica de MLflow, ejecuta en la terminal:

```bash
mlflow ui --port 5000
```

Luego abre http://localhost:5000 en tu navegador.

**Explora la UI** y verifica:
- [ ] Los 3 modelos aparecen como runs separadas
- [ ] Cada run tiene parámetros, métricas y artefactos
- [ ] Puedes comparar runs seleccionando varias y haciendo clic en "Compare"
- [ ] El modelo registrado aparece en la pestaña "Models"

---
## Parte 8: Ejercicios para el alumno

### Ejercicio A: Hiperparámetros alternativos

Entrena un nuevo modelo Random Forest con `n_estimators=200` y `max_depth=15`.
Regístralo como una nueva run. ¿Mejora las métricas? ¿Compensa el mayor
tiempo de entrenamiento y tamaño del modelo?

```python
# Tu código aquí
```

**Resultados**:
- Accuracy: ____
- F1 Score: ____
- Tiempo de entrenamiento: ____
- ¿Merece la pena? ____

### Ejercicio B: Artefactos adicionales

Añade a una de las runs existentes los siguientes artefactos:
1. Matriz de confusión (como imagen PNG)
2. Classification report (como archivo de texto)
3. Una lista de las 10 features más importantes (si el modelo lo soporta)

```python
# Tu código aquí
```

### Ejercicio C: Evaluación de conveniencia (RA4d)

Completa la siguiente tabla comparativa para decidir qué modelo llevar a producción:

| Criterio | Logistic Regression | Random Forest | XGBoost |
|----------|-------------------|---------------|--------|
| Accuracy | | | |
| F1 Score | | | |
| Tiempo entrenamiento | | | |
| Tamaño del modelo | | | |
| Interpretabilidad | Alta | Media | Baja |
| Facilidad de actualización | | | |

**Modelo seleccionado para producción**:
```
______________________________________________________________
```

**Justificación (conecta con RA4d)**:
```
______________________________________________________________
______________________________________________________________
```

### Ejercicio D: Conexión con F2

¿Cómo cambia este proceso si en lugar de un dataset estático
los datos se actualizan semanalmente? ¿Cómo afectaría al Model Registry
y a la comparación de modelos?

```
______________________________________________________________
______________________________________________________________
```

---
## Referencias

### Teoría
- `01-teoria/04_experimentacion_mlflow.md` — Guía teórica de F3
- `01-teoria/03_pipeline_datos.md` — Pipeline de datos (F2)

### UD5/UD6
- `06-llm-agentes/01-teoria/MLflow_Documentacion.md` — Documentación MLflow
- `06-llm-agentes/03-practicas/100_mlflow_llamaindex_rag.ipynb` — MLflow con RAG
- `05-cloud-mlops/01-teoria/03-mlops.md` — MLOps y experimentación

### Documentación
- [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html)
- [MLflow Model Registry](https://mlflow.org/docs/latest/model-registry.html)